<a href="https://colab.research.google.com/github/haidy47-design/spark_upload_queries/blob/main/SparkLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyspark


In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("Retail Lab") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
print("Spark version:", spark.version)

Spark version: 4.0.2


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving customers.json to customers (2).json
Saving sales.csv to sales (2).csv


In [ ]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [ ]:
customers_df = spark.read.json("customers.json")

In [ ]:
sales_df.show(5)
customers_df.show(5)

+--------+-----------+-------+-----------+--------+------+----------+
|order_id|customer_id|product|   category|quantity| price|order_date|
+--------+-----------+-------+-----------+--------+------+----------+
|       1|       C101| Laptop|Electronics|       1|1200.5|2025-01-01|
|       2|       C102|  Mouse|Electronics|       2|  25.0|2025-01-02|
|       3|       C101|  Chair|  Furniture|       1| 150.0|2025-01-03|
|       4|       C103|   Desk|  Furniture|       1| 300.0|2025-01-04|
|       5|       C104|  Phone|Electronics|       2| 500.0|2025-01-05|
+--------+-----------+-------+-----------+--------+------+----------+
only showing top 5 rows
+---+-------+-----------+-----+
|age|country|customer_id| name|
+---+-------+-----------+-----+
| 30|  Egypt|       C101|  Ali|
| 25|    UAE|       C102| Sara|
| 40|    USA|       C103| John|
| 28|  Egypt|       C104| Mona|
| 35|    KSA|       C105|Ahmed|
+---+-------+-----------+-----+
only showing top 5 rows


In [ ]:
sales_df.printSchema()
customers_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)

root
 |-- age: long (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)



In [ ]:
sales_df = sales_df.dropna()
customers_df = customers_df.dropna()

In [ ]:
sales_df.show(5)
customers_df.show(5)

+--------+-----------+-------+-----------+--------+------+----------+
|order_id|customer_id|product|   category|quantity| price|order_date|
+--------+-----------+-------+-----------+--------+------+----------+
|       1|       C101| Laptop|Electronics|       1|1200.5|2025-01-01|
|       2|       C102|  Mouse|Electronics|       2|  25.0|2025-01-02|
|       3|       C101|  Chair|  Furniture|       1| 150.0|2025-01-03|
|       4|       C103|   Desk|  Furniture|       1| 300.0|2025-01-04|
|       5|       C104|  Phone|Electronics|       2| 500.0|2025-01-05|
+--------+-----------+-------+-----------+--------+------+----------+
only showing top 5 rows
+---+-------+-----------+-----+
|age|country|customer_id| name|
+---+-------+-----------+-----+
| 30|  Egypt|       C101|  Ali|
| 25|    UAE|       C102| Sara|
| 40|    USA|       C103| John|
| 28|  Egypt|       C104| Mona|
| 35|    KSA|       C105|Ahmed|
+---+-------+-----------+-----+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn("price", col("price").cast("double")) \
                   .withColumn("quantity", col("quantity").cast("int"))

In [ ]:
sales_df = sales_df.withColumn("total_amount", col("price") * col("quantity"))

In [ ]:
sales_df.orderBy(col("total_amount").desc()).show(5)

+--------+-----------+-------+-----------+--------+------+----------+------------+
|order_id|customer_id|product|   category|quantity| price|order_date|total_amount|
+--------+-----------+-------+-----------+--------+------+----------+------------+
|       1|       C101| Laptop|Electronics|       1|1200.5|2025-01-01|      1200.5|
|       5|       C104|  Phone|Electronics|       2| 500.0|2025-01-05|      1000.0|
|       8|       C106|   Sofa|  Furniture|       1|1000.0|2025-01-08|      1000.0|
|       6|       C105| Tablet|Electronics|       1| 800.0|2025-01-06|       800.0|
|       4|       C103|   Desk|  Furniture|       1| 300.0|2025-01-04|       300.0|
+--------+-----------+-------+-----------+--------+------+----------+------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import sum, count

sales_df.groupBy("category").agg(
    sum("total_amount").alias("total_sales"),
    count("order_id").alias("num_orders")
).show()

+-----------+-----------+----------+
|   category|total_sales|num_orders|
+-----------+-----------+----------+
|Electronics|     3125.5|         5|
|  Furniture|     1450.0|         3|
+-----------+-----------+----------+



In [ ]:
joined_df = sales_df.join(customers_df, on="customer_id", how="inner")

In [ ]:
joined_df.groupBy("country").agg(
    sum("total_amount").alias("total_sales")
).show()

+-------+-----------+
|country|total_sales|
+-------+-----------+
|    USA|      300.0|
|    UAE|      125.0|
|    KSA|      800.0|
|  Egypt|     3350.5|
+-------+-----------+



In [ ]:
joined_df.groupBy("customer_id", "name").agg(
    sum("total_amount").alias("total_spending")
).orderBy(col("total_spending").desc()).show(3)

+-----------+----+--------------+
|customer_id|name|total_spending|
+-----------+----+--------------+
|       C101| Ali|        1350.5|
|       C104|Mona|        1000.0|
|       C106|Lina|        1000.0|
+-----------+----+--------------+
only showing top 3 rows
